In [5]:
!pip install langchain
!pip install langchain-classic
!pip install langchain-ollama
!pip install langchain-chroma
!pip install tensorflow
!pip install langchain-chroma
!pip install langchain_community
!pip install panda -U
!pip install chromadb -U
!pip install -U pypdf

  Using cached opentelemetry_exporter_otlp_proto_common-1.38.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached opentelemetry_proto-1.38.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached opentelemetry_sdk-1.38.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_api-1.38.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_semantic_conventions-0.59b0-py3-none-any.whl.metadata (2.4 kB)
Using cached opentelemetry_exporter_otlp_proto_common-1.38.0-py3-none-any.whl (18 kB)
Using cached opentelemetry_proto-1.38.0-py3-none-any.whl (72 kB)
Using cached opentelemetry_sdk-1.38.0-py3-none-any.whl (132 kB)
Using cached opentelemetry_api-1.38.0-py3-none-any.whl (65 kB)
Using cached opentelemetry_semantic_conventions-0.59b0-py3-none-any.whl (207 kB)
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.37.0
    Uninstalling opentelemetry-proto-1.37.0:
      Successfully uninstalled opentelemetry-proto-1.37.0
  Attempting unin

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

In [3]:
# Define variables
pdf_path = "MS_CSC_Catalog.pdf"
collection_name = "langchain_chroma_grad"
embedding_model = "embeddinggemma" # Or "mxbai-embed-large"
llm_model = "granite4" # Or "llama3"

In [4]:
print("--- Task 1: Initialize Embeddings and Vector Store Setup ---")
embeddings = OllamaEmbeddings(model=embedding_model)

--- Task 1: Initialize Embeddings and Vector Store Setup ---


In [6]:
print(f"--- Task 2: Process {pdf_path} ---")
if not os.path.exists(pdf_path):
    print(f"Error: {pdf_path} not found.")
    exit()

loader = PyPDFLoader(pdf_path)
pages = loader.load()
print(f"Loaded {len(pages)} pages.")


--- Task 2: Process MS_CSC_Catalog.pdf ---
Loaded 4 pages.


In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " "]
)
splits = text_splitter.split_documents(pages)
print(f"Split into {len(splits)} chunks.")

Split into 12 chunks.


In [8]:
print("--- Task 3: Add documents to Chroma Vector Store ---")
    # Generate explicit IDs
ids = [str(i) for i in range(len(splits))]
    
    # Initialize Chroma and add documents    
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name=collection_name,
    ids=ids
    # persist_directory="./chroma_db" # Optional: Uncomment to persist
)
print(f"Added {len(splits)} documents to collection '{collection_name}'.")

--- Task 3: Add documents to Chroma Vector Store ---
Added 12 documents to collection 'langchain_chroma_grad'.


In [9]:
print("\n--- Task 4: Retrieve all documents ---")
all_docs = vectorstore.get()
print(f"Retrieved {len(all_docs['ids'])} documents.")
print(all_docs) # Commented out to avoid cluttering output, but this is the retrieval.

# 5. Use get() to retrieve page 0 information.
print("\n--- Task 5: Retrieve page 0 information ---")
page_0_docs = vectorstore.get(where={"page": 0})
print(f"Found {len(page_0_docs['ids'])} chunks from page 0.")
if page_0_docs['ids']:
    print(f"First chunk of page 0 content preview: {page_0_docs['documents'][0][:100]}...")

# 6. Use get() to retrieve id 0.
print("\n--- Task 6: Retrieve ID 0 ---")
id_0_doc = vectorstore.get(ids=["0"])
print(f"ID 0 Content: {id_0_doc['documents'][0][:100]}...")


--- Task 4: Retrieve all documents ---
Retrieved 12 documents.
{'ids': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11'], 'embeddings': None, 'documents': ["MS in Computer Science\uf0c9 2025-2026 Edition\nTotal units required for MS: 30\nProgram Description\nThe Computer Science Department offers Master's Degree programs in Computer Science and Software Engineering, Certificates\nof Advanced Study for students enrolled in the Computer Science program, and a Master's Degree joint program in Computer\nEngineering.\nThe primary goal of each of these programs is to prepare students to serve as effective professional computer specialists in a\nsociety which increasingly depends on computer usage and technology.\nA secondary goal is to prepare interested students for research, teaching, or further study toward the Ph.D. in Computer Science.\nThe programs also enable individuals with background in other areas to obtain the skills and knowledge necessary to enter and\nadvance in 

In [10]:
print("\n--- Task 7: Update content of ID 0 ---")
prefix_text = "This document is the Catalog of Master Program (MS) in Computer Science. "
current_content = id_0_doc['documents'][0]
new_content = prefix_text + current_content

# Update using the vectorstore's update_document method
current_metadata = id_0_doc['metadatas'][0]
updated_doc = Document(page_content=new_content, metadata=current_metadata)
vectorstore.update_document(document_id="0", document=updated_doc)
print("Updated ID 0.")
updated_id_0 = vectorstore.get(ids=["0"])
print(f"New ID 0 Content Start: {updated_id_0['documents'][0][:100]}...")



--- Task 7: Update content of ID 0 ---
Updated ID 0.
New ID 0 Content Start: This document is the Catalog of Master Program (MS) in Computer Science. MS in Computer Science 202...


In [11]:
print("\n--- Task 8: Similarity Search with Relevance Scores (k=4) ---")
query = "what are the admission requirements?"
results_with_scores = vectorstore.similarity_search_with_relevance_scores(query, k=4)
print(f"Query: {query}")
for doc, score in results_with_scores:
    print(f"[Score: {score:.4f}] {doc.page_content[:100]}...")


--- Task 8: Similarity Search with Relevance Scores (k=4) ---
Query: what are the admission requirements?
[Score: 0.3513] website:
an online application for admission;
two sets of official transcripts from all colleges and...
[Score: 0.2576] a baccalaureate degree;
a minimum 3.0 GPA in the last 60 units attempted;
GRE general test;
mathemat...
[Score: 0.2563] CSC 35 Introduction to Computer Architecture 3
CSC 60 Introduction to Systems Programming in UNIX 3
...
[Score: 0.2323] areas: computer architecture/computer engineering, database management systems, information assuranc...


In [ ]:
print("\n--- Task 9: Similarity Search with Cutoff Score ---")
cutoff = 0.25
print(f"Applying cutoff score: {cutoff}")
filtered_results = [
    (doc, score) for doc, score in results_with_scores 
    if score >= cutoff
]
    
if not filtered_results:
    print("No results met the cutoff score.")
else:
    for doc, score in filtered_results:
        print(f"[Score: {score:.4f}] {doc.page_content[:50]}...")



--- Task 9: Similarity Search with Cutoff Score ---
Applying cutoff score: 0.25
[Score: 0.3513] website:
an online application for admission;
two ...
[Score: 0.2576] a baccalaureate degree;
a minimum 3.0 GPA in the l...
[Score: 0.2563] CSC 35 Introduction to Computer Architecture 3
CSC...


In [16]:
print("\n--- Task 10: RAG with ChatPromptTemplate ---")
llm = OllamaLLM(model=llm_model, temperature=0)

template = """
Answer the question based only on the following context: 
{context}
Question: 
{question}
"""
prompt = ChatPromptTemplate.from_template(template)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_response = chain.invoke(query)
print(f"Question: {query}")
print(f"Answer: {rag_response}")


--- Task 10: RAG with ChatPromptTemplate ---
Question: what are the admission requirements?
Answer: Based on the provided context, the key admission requirements for the MS in Computer Science program at Sacramento State are:

1. A baccalaureate degree.

2. A minimum 3.0 GPA in the last 60 units attempted (completed within seven years prior to graduation).

3. GRE general test scores.

4. Mathematical preparation including:
   - Two semesters of calculus
   - One semester of calculus-based probability and statistics

5. Computer Science lower-division preparation corresponding to Sacramento State courses, which includes:
   - MATH 30 Calculus I (4 units)
   - MATH 31 Calculus II (4 units) 
   - STAT 50 Introduction to Probability and Statistics (4 units)

6. Programming proficiency, discrete structures, machine organization, and UNIX/PC-based program development environment proficiency.

7. Computer Science advanced preparation as evidenced by:
   - A 3.25 GPA in upper division Sacram

In [17]:
print("\n--- Task 11: RAG with RetrievalQA ---")
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

qa_response = qa_chain.invoke(query)
print(f"Question: {query}")
print(f"Answer: {qa_response['result']}")


--- Task 11: RAG with RetrievalQA ---
Question: what are the admission requirements?
Answer: Based on the information provided, the key admission requirements for the MS in Computer Science program at Sacramento State are:

1. A baccalaureate degree

2. A minimum GPA of 3.0 in the last 60 units attempted (completed within seven years prior to graduation)

3. GRE general test score

4. Mathematical preparation including:
   - Two semesters of calculus 
   - One semester of calculus-based probability and statistics
   Corresponding to Sacramento State courses: MATH 30 Calculus I, MATH 31 Calculus II, STAT 50 Introduction to Probability and Statistics

5. Computer Science lower-division preparation including programming proficiency, discrete structures, machine organization, and UNIX/PC-based program development environment proficiency (corresponding to Sacramento State courses CSC 15 Programming Concepts and Methodology I, CSC 20 Programming Concepts and Methodology II, CSC 28 Discrete 

In [18]:
print("\n--- Task 12: Run 3 other queries ---")
queries = [
    "What is the minimum GPA requirement?",
    "How many units are required for the master's degree?",
    "What are the Software Engineering courses provided?"
]

for q in queries:
    print(f"\nQuery: {q}")
    res = qa_chain.invoke(q)
    print(f"Answer: {res['result']}")


--- Task 12: Run 3 other queries ---

Query: What is the minimum GPA requirement?
Answer: According to the context provided on the website, the minimum cumulative GPA required for admission to the MS in Computer Science program at Sacramento State is 3.0. It also states that no grade below "C" may count toward the degree.

Therefore, the minimum GPA requirement is a 3.0 cumulative GPA.

Query: How many units are required for the master's degree?
Answer: According to the information provided in the Catalog of Master Program (MS) in Computer Science, a total of 30 units are required for the MS degree.

Query: What are the Software Engineering courses provided?
Answer: Based on the context provided, the Software Engineering courses offered at Sacramento State for the MS in Computer Science program include:

1. CSC 230 - Software System Engineering
2. CSC 231 - Software Engineering Metrics
3. CSC 232 - Software Requirements Analysis and Design
4. CSC 233 - Advanced Software Engineering Pr

In [19]:
print("\n--- Task 13: Delete ID 0 ---")
vectorstore.delete(ids=["0"])

# Verify
deleted_check = vectorstore.get(ids=["0"])
if not deleted_check['ids']:
    print("Successfully deleted ID 0 (not found in store).")
else:
    print("Error: ID 0 still exists.")



--- Task 13: Delete ID 0 ---
Successfully deleted ID 0 (not found in store).
